# Семинар 3. Конверсия выросла, а чекаут течёт

**Вопрос недели:** что мерить, чтобы видеть здоровье бизнеса — и чтобы ML-фича не росла за счёт денег?

**Кейс.** Понедельник, 13 апреля 2026. На дашборде «Лукошко» конверсия корзина → заказ за восемь недель выросла с 48% до 53%. Директор по продукту просит слайд «что мы сделали правильно» и план, куда расти дальше. Одновременно в чате появляется сообщение: «по воронке чекаут — самое узкое место, давайте переделаем его целиком». Ваша задача — проверить цифры до того, как они уйдут в презентацию.

Пять частей:
1. **Что именно выросло** (≈ 15 мин): дубли в логе, конверсия до и после.
2. **Источники трафика и парадокс Симпсона** (≈ 25 мин): состав или внутри групп; деньги вместо конверсии.
3. **«Чекаут — самое узкое место»?** (≈ 15 мин): что сломано в логе.
4. **Три гипотезы роста** (≈ 10 мин): для каждой — почему это может быть не причина.
5. **Дерево метрик «Лукошко» до CM** (≈ 25 мин): ветки, владельцы, guardrail с порогами для будущей персональной главной.

**Треки.** 🟢 *Базовый* — запускайте готовые ячейки и отвечайте на вопросы. 🔵 *Глубокий* — ячейки с пометкой «глубокий трек» сначала напишите сами, потом сверьтесь с готовым кодом.

Работаем в DuckDB только на чтение; всё, что строим, — временные таблицы.

In [ ]:
import pathlib
import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)

ROOT = pathlib.Path.cwd().resolve().parents[1] / "практика"   # папка ноутбука → корень репозитория → практика
con = duckdb.connect(str(ROOT / "data" / "lukoshko.duckdb"), read_only=True)
q = lambda sql: con.execute(sql).fetchdf()
SCALE = con.execute("SELECT value FROM meta.scale WHERE param = 'scale_factor'").fetchone()[0]
print(f"Множитель до масштаба бизнеса: {SCALE}")
q("SELECT MIN(event_ts) AS с, MAX(event_ts) AS по, COUNT(*) AS событий, COUNT(DISTINCT session_id) AS сессий FROM raw.events")

## Часть 1. Что именно выросло

Единица анализа — **сессия**: одна строка на `session_id`. Признаки сессии: источник (`utm_source` из `session_start`), была ли корзина, дошла ли до `checkout_view`, создан ли заказ. Сначала смотрим так, как видит дашборд.

In [ ]:
con.execute('''
CREATE OR REPLACE TEMP TABLE sess_raw AS
SELECT session_id, MIN(event_ts) AS t,
       MAX((event_name = 'add_to_cart')::INT)    AS cart,
       MAX((event_name = 'order_created')::INT)  AS ord
FROM raw.events GROUP BY session_id
''')
weeks = q('''
SELECT date_trunc('week', t)::DATE AS неделя,
       SUM(cart) AS "сессий с корзиной",
       ROUND(100 * AVG(ord) FILTER (cart = 1), 1) AS "корзина → заказ, %"
FROM sess_raw GROUP BY 1 ORDER BY 1
''')
weeks

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(weeks["неделя"].astype(str).str[5:], weeks["корзина → заказ, %"], marker="o", color="#E97132")
ax.set_ylabel("корзина → заказ, %"); ax.set_title("Так это видит дашборд"); ax.grid(alpha=.3)
plt.show()

### 1a. Дубли событий

Ключ `event_id` должен быть уникальным. Проверяем и заодно смотрим, как дубли влияют на конверсию, посчитанную **по числу событий** (так её часто считают в BI).

In [ ]:
q('''
SELECT platform AS платформа, COUNT(*) AS строк, COUNT(DISTINCT event_id) AS событий,
       COUNT(*) - COUNT(DISTINCT event_id) AS дублей,
       ROUND(100.0 * (COUNT(*) - COUNT(DISTINCT event_id)) / COUNT(*), 2) AS "дублей, %"
FROM raw.events GROUP BY platform ORDER BY платформа
''')

In [ ]:
# «заказ / добавление в корзину» по числу событий — без дедупликации и с ней
q('''
WITH d AS (SELECT DISTINCT ON (event_id) * FROM raw.events)
SELECT 'без дедупликации' AS вариант, COUNT(*) FILTER (event_name = 'order_created') AS заказов,
       COUNT(*) FILTER (event_name = 'add_to_cart') AS добавлений,
       ROUND(100.0 * COUNT(*) FILTER (event_name = 'order_created') / COUNT(*) FILTER (event_name = 'add_to_cart'), 2) AS "заказ / добавление, %"
FROM raw.events
UNION ALL
SELECT 'по event_id', COUNT(*) FILTER (event_name = 'order_created'), COUNT(*) FILTER (event_name = 'add_to_cart'),
       ROUND(100.0 * COUNT(*) FILTER (event_name = 'order_created') / COUNT(*) FILTER (event_name = 'add_to_cart'), 2)
FROM d
UNION ALL
SELECT 'заказы из OLTP (raw.orders)', COUNT(*), NULL, NULL FROM raw.orders
WHERE created_at >= '2026-02-16' AND created_at < '2026-04-13'
''')

**Вопросы.**
1. На какой платформе дубли и сколько заказов «по событиям» лишние? Какой источник правды для числа заказов?
2. Меняют ли дубли конверсию на уровне **сессии** (флаг «в сессии был заказ»)? А показатель «заказов на сессию»?
3. 🔵 Запишите правило дедупликации одной строкой: ключ, какая копия остаётся, в каком слое это делается.

Дальше работаем с дедуплицированным логом.

In [ ]:
con.execute('''
CREATE OR REPLACE TEMP TABLE sess AS
WITH e AS (SELECT DISTINCT ON (event_id) * FROM raw.events)
SELECT session_id, ANY_VALUE(platform) AS platform, ANY_VALUE(app_version) AS ver, MIN(event_ts) AS t,
       MAX(CASE WHEN event_name = 'session_start' THEN json_extract_string(properties, '$.utm_source') END) AS src,
       MAX((event_name = 'add_to_cart')::INT)    AS cart,
       MAX((event_name = 'checkout_view')::INT)  AS chk,
       MAX((event_name = 'order_created')::INT)  AS ord,
       MAX(CASE WHEN event_name = 'order_created' THEN json_extract(properties, '$.order_id')::BIGINT END) AS order_id
FROM e GROUP BY session_id
''')
con.execute('''
CREATE OR REPLACE TEMP TABLE carts AS
SELECT s.*, (src = 'ctx_brand') AS brand, o.gmv, o.cm,
       CASE WHEN t < '2026-03-01' THEN 'до' WHEN t >= '2026-03-23' THEN 'после' END AS period
FROM sess s LEFT JOIN mart.orders o USING (order_id)
WHERE cart = 1
''')
# «до» — две недели 16.02–01.03, «после» — три недели 23.03–12.04
q('''
SELECT period AS период, COUNT(*) AS "сессий с корзиной", ROUND(100 * AVG(ord), 1) AS "корзина → заказ, %",
       ROUND(100 * AVG(brand::INT), 1) AS "доля брендового контекста, %"
FROM carts WHERE period IS NOT NULL GROUP BY 1 ORDER BY 1 DESC
''')

Конверсия выросла: примерно +5 п.п. Сколько это в деньгах? В паспорте «Лукошко»: **+1 п.п. конверсии корзина → заказ ≈ +13 млн ₽ CM в месяц**, если чек и доля возвратов не изменились. Первая реакция — «+68 млн ₽ в месяц». Проверяем, что стоит за ростом.

## Часть 2. Источники трафика и парадокс Симпсона

In [ ]:
by_src = q('''
SELECT src AS источник,
       COUNT(*) FILTER (period = 'до')    AS "корзин до",
       COUNT(*) FILTER (period = 'после') AS "корзин после",
       ROUND(100 * AVG(ord) FILTER (period = 'до'), 1)    AS "конверсия до, %",
       ROUND(100 * AVG(ord) FILTER (period = 'после'), 1) AS "конверсия после, %"
FROM carts WHERE period IS NOT NULL GROUP BY 1 ORDER BY 2 DESC
''')
by_src["Δ, п.п."] = (by_src["конверсия после, %"] - by_src["конверсия до, %"]).round(1)
by_src["доля до, %"] = (100 * by_src["корзин до"] / by_src["корзин до"].sum()).round(1)
by_src["доля после, %"] = (100 * by_src["корзин после"] / by_src["корзин после"].sum()).round(1)
by_src

In [ ]:
# недельная картина: доля бренда и конверсия внутри групп
wk = q('''
SELECT date_trunc('week', t)::DATE AS неделя,
       ROUND(100 * AVG(ord), 1)                                  AS "все",
       ROUND(100 * AVG(ord) FILTER (brand), 1)                   AS "бренд",
       ROUND(100 * AVG(ord) FILTER (NOT brand), 1)               AS "остальные",
       ROUND(100 * AVG(brand::INT), 1)                           AS "доля бренда, %"
FROM carts GROUP BY 1 ORDER BY 1
''')
print(wk)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4))
x = wk["неделя"].astype(str).str[5:]
a1.plot(x, wk["все"], marker="o", color="#E97132", label="все")
a1.plot(x, wk["бренд"], marker="o", color="#0E2841", label="бренд")
a1.plot(x, wk["остальные"], marker="o", color="#5B7DB1", label="остальные")
a1.set_title("Конверсия корзина → заказ, %"); a1.legend(); a1.grid(alpha=.3)
a2.bar(x, wk["доля бренда, %"], color="#B9C2CC"); a2.set_title("Доля брендового контекста в корзинах, %"); a2.grid(alpha=.3)
plt.tight_layout(); plt.show()

### Разложение: состав против «внутри»

Изменение общей конверсии = **эффект состава** (доли групп сменились, конверсии как «до») + **эффект внутри групп** (доли как «после», конверсии сменились).

**Задание (🔵 глубокий трек — сначала сами).** Посчитайте оба слагаемых по двум группам (бренд / не бренд). Сумма должна дать общий прирост.

In [ ]:
g = q('''
SELECT brand, period, COUNT(*) AS n, AVG(ord) AS conv
FROM carts WHERE period IS NOT NULL GROUP BY 1, 2
''')
p = g.pivot(index="brand", columns="period", values=["n", "conv"])
w0, w1 = p["n"]["до"] / p["n"]["до"].sum(), p["n"]["после"] / p["n"]["после"].sum()
c0, c1 = p["conv"]["до"], p["conv"]["после"]
total  = (w1 * c1).sum() - (w0 * c0).sum()
mix    = ((w1 - w0) * c0).sum()
within = (w1 * (c1 - c0)).sum()
fixed  = (w0 * c1).sum()            # конверсия «после» при составе «до»
print(f"Общий прирост:          {100 * total:+.1f} п.п.")
print(f"  из них состав:        {100 * mix:+.1f} п.п.")
print(f"  из них внутри групп:  {100 * within:+.1f} п.п.")
print(f"Конверсия после при составе «до»: {100 * fixed:.1f}% (было {100 * (w0 * c0).sum():.1f}%)")

In [ ]:
# насколько можно доверять знаку «внутри»? бутстрэп по сессиям: прирост при фиксированном составе «до»
rng = np.random.default_rng(42)
d = q("SELECT brand::INT AS brand, period, ord FROM carts WHERE period IS NOT NULL")
arr = {(b, per): d[(d.brand == b) & (d.period == per)]["ord"].to_numpy() for b in (0, 1) for per in ("до", "после")}
w_pre = np.array([len(arr[(b, "до")]) for b in (0, 1)]) / sum(len(arr[(b, "до")]) for b in (0, 1))
res = []
for _ in range(500):
    c_pre  = [rng.choice(arr[(b, "до")],    len(arr[(b, "до")])).mean()    for b in (0, 1)]
    c_post = [rng.choice(arr[(b, "после")], len(arr[(b, "после")])).mean() for b in (0, 1)]
    res.append(((np.array(c_post) - np.array(c_pre)) * w_pre).sum())
lo, hi = np.percentile(res, [2.5, 97.5])
print(f"Прирост при составе «до»: {100 * np.mean(res):+.1f} п.п., 95% интервал [{100 * lo:+.1f}; {100 * hi:+.1f}]")

### Деньги: конверсия — не CM

Для заказов из лога есть `order_id` → присоединяем `mart.orders` (GMV и CM заказа). Считаем **CM на сессию с корзиной**: конверсия × CM заказа ближе к деньгам, чем одна конверсия.

In [ ]:
m = q('''
SELECT period AS период, brand AS бренд, COUNT(*) AS n,
       ROUND(100 * AVG(ord), 1)           AS "конверсия, %",
       ROUND(SUM(gmv) / SUM(ord))         AS "чек, ₽",
       ROUND(SUM(cm) / SUM(ord))          AS "CM заказа, ₽",
       ROUND(SUM(cm) / COUNT(*), 1)       AS "CM на корзину, ₽"
FROM carts WHERE period IS NOT NULL GROUP BY ALL ORDER BY бренд, период DESC
''')
print(m)
tot = q('''SELECT period AS период, ROUND(SUM(cm) / COUNT(*), 1) AS "CM на корзину, ₽", ROUND(SUM(gmv) / SUM(ord)) AS "чек, ₽"
           FROM carts WHERE period IS NOT NULL GROUP BY 1 ORDER BY 1 DESC''')
print(tot)
cm_cart = q("SELECT brand, period, SUM(cm) / COUNT(*) AS v FROM carts WHERE period IS NOT NULL GROUP BY 1, 2").pivot(index="brand", columns="period", values="v")
print(f"CM на корзину при составе «до»: {(w0 * cm_cart['после']).sum():.1f} ₽ против {(w0 * cm_cart['до']).sum():.1f} ₽ раньше")
gain_fixed = fixed - (w0 * c0).sum()
print(f"Грубая оценка по правилу «+1 п.п. ≈ 13 млн ₽ CM в месяц»: по дашборду {100*total:+.1f} п.п. ≈ {13*100*total:+.0f} млн ₽; "
      f"при неизменном составе {100*gain_fixed:+.1f} п.п. ≈ {13*100*gain_fixed:+.0f} млн ₽")

**Вопросы к части 2.**
1. Какую долю роста объясняет состав? Что случилось с конверсией внутри каждого источника — в скольких из шести она упала? Какие из этих падений вы бы назвали статистически надёжными, а какие нет?
2. Почему конверсия брендового контекста выше всех остальных? Означает ли это, что «брендовую рекламу надо расширять»? (подсказка: кто вводит в поисковик название «Лукошко»?)
3. Какую цифру вы положите на слайд директору вместо «48% → 53%»?
4. 🔵 Где ещё в «Лукошко» состав меняется между периодами так же (платформы, зоны, новички и вернувшиеся, акции)? Как сделать так, чтобы дашборд показывал состав рядом со средним автоматически?

## Часть 3. «Чекаут — самое узкое место»?

In [ ]:
fn = q('''
SELECT date_trunc('week', t)::DATE AS неделя,
       ROUND(100 * AVG(chk) FILTER (cart = 1), 1)                  AS "корзина → checkout_view, %",
       ROUND(100 * AVG(ord) FILTER (chk = 1), 1)                   AS "checkout_view → заказ, %",
       ROUND(100 * AVG(ord) FILTER (cart = 1), 1)                  AS "корзина → заказ, %"
FROM sess GROUP BY 1 ORDER BY 1
''')
fn

In [ ]:
# по платформам и версиям: где пропал шаг?
q('''
SELECT platform AS платформа, ver AS версия, MIN(t)::DATE AS с, MAX(t)::DATE AS по,
       COUNT(*) FILTER (ord = 1) AS заказов,
       ROUND(100 * AVG(chk) FILTER (cart = 1), 1) AS "корзина → checkout_view, %",
       ROUND(100 * (1 - AVG(chk) FILTER (ord = 1)), 1) AS "заказов БЕЗ checkout_view, %"
FROM sess GROUP BY 1, 2 ORDER BY 1, 2
''')

In [ ]:
# недели по iOS: заказ без показа чекаута невозможен по логике продукта
q('''
SELECT date_trunc('week', t)::DATE AS неделя,
       COUNT(*) FILTER (ord = 1) AS "заказов iOS",
       COUNT(*) FILTER (ord = 1 AND chk = 0) AS "из них без checkout_view",
       COUNT(*) FILTER (ord = 1 AND chk = 0 AND ver = '7.0.0') AS "из них версия 7.0.0"
FROM sess WHERE platform = 'ios' GROUP BY 1 ORDER BY 1
''')

**Вопросы к части 3.**
1. Что показывает воронка «корзина → checkout_view → заказ» по неделям? Какой вывод сделает менеджер, не заглянувший в версии?
2. С какой даты и на какой платформе шаг пропал? Что произошло 6 апреля? Какую долю корзин это затрагивает?
3. Почему «корзина → заказ» (первая и последняя точки) не пострадала, а промежуточные шаги — да? Какое правило отсюда: что мерить в первую очередь на критичном пути?
4. 🔵 Какой тест в чистом слое поймал бы пропажу за сутки? (подсказка: «заказ без предшествующего checkout_view» — невозможное событие.) Что делать с этими тремя неделями в истории: пересчитать, пометить, исключить?

## Часть 4. Три гипотезы роста

Заполните таблицу в чате команды (шаблон — ниже). Для каждой гипотезы: что делаем; какую метрику ждём; **почему это может быть не причина** того, что мы видели; как проверить.

| № | Гипотеза роста | Метрика | Почему может быть не причина | Как проверить |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |

Подсказки для разогрева: «упростим чекаут»; «расширим брендовый контекст»; «скидка в корзине». Вспомните, что вы видели в частях 2 и 3: состав, логирование, дубли.

## Часть 5. Дерево метрик «Лукошко» до CM

Берём последний полный месяц выборки (август 2026) и раскладываем деньги на множители. Выборка — 80 000 новых пользователей; в масштабе бизнеса умножаем на `scale_factor`.

In [ ]:
t = q('''
SELECT COUNT(DISTINCT user_id) AS buyers, COUNT(*) AS orders, SUM(gmv) AS gmv, SUM(platform_revenue) AS revenue,
       SUM(logistics_cost) AS logistics, SUM(promo_cost) AS promo, SUM(acquiring_cost) AS acquiring,
       SUM(reversed_revenue) AS reversed, SUM(return_cost) AS return_cost, SUM(cm) AS cm
FROM mart.orders WHERE order_ts >= '2026-08-01' AND order_ts < '2026-09-01'
''').iloc[0]
tree = pd.DataFrame({
    "узел дерева": ["покупателей в месяц", "заказов на покупателя (NSM)", "средний чек, ₽", "доход платформы, % GMV",
                    "логистика на заказ, ₽", "промо, % GMV", "сторно дохода и стоимость возвратов, % GMV", "CM, % GMV", "CM на заказ, ₽"],
    "значение": [t.buyers, t.orders / t.buyers, t.gmv / t.orders, 100 * t.revenue / t.gmv, t.logistics / t.orders,
                 100 * t.promo / t.gmv, 100 * (t.reversed + t.return_cost) / t.gmv, 100 * t.cm / t.gmv, t.cm / t.orders]}).round(2)
print(tree.to_string(index=False))
print(f"\nGMV месяца в выборке {t.gmv/1e6:.1f} млн ₽ → в масштабе бизнеса ≈ {t.gmv*SCALE/1e9:.2f} млрд ₽ (паспорт: ≈ 7,0)")
print(f"CM месяца в масштабе бизнеса ≈ {t.cm*SCALE/1e6:.0f} млн ₽ (паспорт: ≈ 630)")

In [ ]:
# чувствительность: на сколько процентов вырастет CM месяца от +1% каждого множителя при прочих равных
buyers, opb, aov = t.buyers, t.orders / t.buyers, t.gmv / t.orders
take = t.revenue / t.gmv
other = (t.promo + t.acquiring + t.reversed + t.return_cost) / t.gmv      # промо, эквайринг, возвраты — доля GMV
def cm_model(buyers=buyers, opb=opb, aov=aov, take=take, log=t.logistics / t.orders):
    orders = buyers * opb
    return orders * (aov * take - log) - orders * aov * other
base = cm_model()
rows = {"покупателей": cm_model(buyers=buyers * 1.01), "заказов на покупателя": cm_model(opb=opb * 1.01),
        "чек": cm_model(aov=aov * 1.01), "доход платформы, % GMV": cm_model(take=take * 1.01),
        "логистика на заказ": cm_model(log=t.logistics / t.orders * 1.01)}
print(f"Сверка модели с фактом: {base/1e6:.3f} против {t.cm/1e6:.3f} млн ₽")
pd.DataFrame({"+1% к множителю → CM, %": {k: round(100 * (v / base - 1), 2) for k, v in rows.items()}})

In [ ]:
# шум недельных guardrail-метрик (13 полных недель июнь–август; пороги не «с потолка»): возвраты и CM на заказ
q('''
SELECT date_trunc('week', order_ts)::DATE AS неделя, COUNT(*) AS заказов,
       ROUND(100 * AVG(has_return::INT), 2) AS "заказов с возвратом, %",
       ROUND(SUM(cm) / COUNT(*), 1)         AS "CM на заказ, ₽"
FROM mart.orders WHERE order_ts >= '2026-06-01' AND order_ts < '2026-08-31' GROUP BY 1 ORDER BY 1
''').describe().loc[["mean", "std", "min", "max"]].round(2)

**Задание (групповая работа, шаблон на листе или в Miro).**

1. Постройте дерево метрик «Лукошко» от **NSM** (заказов в месяц на активного покупателя) до **CM**: минимум три уровня, у каждой ветки — владелец (кто отвечает), тип (input / output) и источник числа.
2. Для **персональной главной** (неделя 10) выберите: одну **proxy-метрику**, **OEC** (одно число для решения) и **2–3 guardrail с числовыми порогами**. Пороги обоснуйте разбросом из ячейки выше или паспортом.
3. Отметьте одну метрику из дерева, которую проще всего «нагеймить» (улучшить, не создав ценности), и скажите чем.

**Вопросы.**
1. Какой множитель дерева самый «дорогой» по чувствительности CM и какая команда за него отвечает? Совпадает ли это с тем, что команда персональной главной будет двигать?
2. Две ML-фичи на главной выросли по CTR на +5%. Какой guardrail из вашего списка первым скажет «стоп», и при каком значении?
3. 🔵 Почему «Лукошко» считает NSM в заказах на покупателя, а не в GMV или выручке? (вспомните «Лигу Ставок»: что происходит с командами, когда north star — деньги.)

## Итог семинара: сообщение директору по продукту (полстраницы)

(1) Что на самом деле произошло с конверсией корзина → заказ: цифры «до» и «после», состав и внутри групп; (2) что сломано в логе и как это исправить в tracking plan; (3) три гипотезы роста с оговорками; (4) дерево метрик и guardrail персональной главной одной таблицей.